# 01 — Setup Credentials (GCP)

Validate gcloud auth, confirm SDK versions, and initialize `config.json` for downstream notebooks.

AWS-side equivalent: `notebooks/aws/01-setup-credentials.ipynb` (AWS SSO + boto3).

## Step 1: Confirm gcloud is authenticated

In [ ]:
import subprocess
out = subprocess.run(['gcloud', 'auth', 'list', '--format=value(account,status)'], capture_output=True, text=True)
print(out.stdout or '(no accounts)')
if out.returncode != 0:
    print('STDERR:', out.stderr)

## Step 2: Confirm ADC (Application Default Credentials) are available

The Python SDKs (Vertex AI, Discovery Engine, etc.) look for ADC. If this fails, run:

```
gcloud auth application-default login
```

In [ ]:
from google.auth import default
from google.auth.exceptions import DefaultCredentialsError
try:
    creds, project = default()
    print(f'✓ ADC found; default project = {project}')
except DefaultCredentialsError as e:
    print(f'✗ ADC missing: {e}')
    print('  → run `gcloud auth application-default login` in your host shell')

## Step 3: Active project + region

In [ ]:
import os
PROJECT_ID = os.environ.get('GCP_PROJECT_ID') or subprocess.run(
    ['gcloud', 'config', 'get-value', 'project'], capture_output=True, text=True
).stdout.strip()
REGION = os.environ.get('GCP_REGION', 'us-central1')
print(f'project: {PROJECT_ID}')
print(f'region : {REGION}')
assert PROJECT_ID, 'GCP_PROJECT_ID is unset and gcloud has no active project — set one before continuing.'

## Step 4: Check required APIs are enabled

In [ ]:
REQUIRED = [
    'aiplatform.googleapis.com',
    'discoveryengine.googleapis.com',
    'alloydb.googleapis.com',
    'cloudtrace.googleapis.com',
    'modelarmor.googleapis.com',
    'storage.googleapis.com',
    'secretmanager.googleapis.com',
]
enabled = subprocess.run(
    ['gcloud', 'services', 'list', '--enabled', '--format=value(name)'],
    capture_output=True, text=True,
).stdout.split()
missing = [s for s in REQUIRED if s not in enabled]
if missing:
    print('✗ Missing APIs (enable with `gcloud services enable ...`):')
    for m in missing: print(f'   {m}')
    print('\nOne-liner:')
    print('   gcloud services enable ' + ' '.join(missing))
else:
    print('✓ All required APIs enabled.')

## Step 5: Validate SDK versions

In [ ]:
import importlib.metadata as md
for pkg in [
    'google-cloud-aiplatform',
    'vertexai',
    'google-genai',
    'google-cloud-discoveryengine',
    'google-cloud-alloydb',
    'google-cloud-alloydb-connector',
    'google-cloud-storage',
    'psycopg',
    'pgvector',
]:
    try:
        print(f'  {pkg:35s} {md.version(pkg)}')
    except md.PackageNotFoundError:
        print(f'  {pkg:35s} (not installed)')

## Step 6: Initialize / update config.json

Writes the GCP context under `cfg['gcp']` so downstream notebooks can read it.

Defaults match the GA-stable Gemini family as of May 2026. Override by editing
`config.json` after this cell runs.

In [ ]:
import json
from pathlib import Path

config_path = Path('config.json')
cfg = json.loads(config_path.read_text()) if config_path.exists() else {}
cfg.setdefault('gcp', {})
cfg['gcp']['project_id'] = PROJECT_ID
cfg['gcp']['region'] = REGION

# Default models — downstream notebooks read these by name so a single edit
# here flips the whole workshop.
cfg['gcp'].setdefault('model_id', 'gemini-2.5-flash')
cfg['gcp'].setdefault('model_id_pro', 'gemini-2.5-pro')
cfg['gcp'].setdefault('reasoning_model', 'gemini-2.5-pro')
cfg['gcp'].setdefault('fast_model', 'gemini-2.5-flash')
cfg['gcp'].setdefault('embedding_model', 'gemini-embedding-001')
# gemini-embedding-001 native dim is 3072; 1536 is a good default (retrieval
# quality matches 3072 within ~1% for short text, half the storage).
cfg['gcp'].setdefault('embedding_dim', 1536)

# Resource handles populated by later notebooks; declared here so reads
# in 14-scoring / 15-deploy / 16-eval / 18b-text-to-sql don't KeyError.
cfg['gcp'].setdefault('staging_bucket', f'gs://{PROJECT_ID}-orgiq-staging')
cfg['gcp'].setdefault('alloydb_cluster', 'orgiq-alloydb')
cfg['gcp'].setdefault('alloydb_instance', 'orgiq-alloydb-primary')
cfg['gcp'].setdefault('alloydb_database', 'orgiq')
cfg['gcp'].setdefault('alloydb_user', 'orgiq')
cfg['gcp'].setdefault('bq_scoring_dataset', 'orgiq_scoring')

cfg['gcp'].setdefault('tracked_resources', [])

config_path.write_text(json.dumps(cfg, indent=2))
print('✓ config.json updated:')
for k, v in cfg['gcp'].items():
    if k == 'tracked_resources':
        print(f'  {k:20s} ({len(v)} entries)')
    else:
        print(f'  {k:20s} {v}')


## Step 7: Smoke-test Vertex AI

Calls the embedding model with a tiny input via the new `google-genai`
client. Confirms ADC + API enablement + region in one shot.

We use the new SDK pattern (`from google import genai`); the legacy
`vertexai.language_models.TextEmbeddingModel` path still works but is
no longer the recommended surface.

In [ ]:
from google import genai
from google.genai.types import EmbedContentConfig

# vertexai=True routes to Vertex AI (regional, your project) rather than the
# public ai.google.dev endpoint. Required for production / IAM-scoped usage.
client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

EMBED_MODEL = cfg['gcp']['embedding_model']
EMBED_DIM = cfg['gcp']['embedding_dim']

resp = client.models.embed_content(
    model=EMBED_MODEL,
    contents=['OrgIQ is an organizational intelligence platform.'],
    config=EmbedContentConfig(
        task_type='RETRIEVAL_DOCUMENT',
        output_dimensionality=EMBED_DIM,
    ),
)
vec = resp.embeddings[0].values
print(f'✓ embedding ok — model={EMBED_MODEL} dim={len(vec)} (requested {EMBED_DIM})')

## Step 8: Smoke-test Gemini text generation

Same `genai.Client` — confirms the chat / generate path works with the
default model. Tiny prompt to keep cost negligible.

In [ ]:
MODEL_ID = cfg['gcp']['model_id']
resp = client.models.generate_content(
    model=MODEL_ID,
    contents='Reply with exactly the word OK.',
)
print(f'✓ {MODEL_ID} responded: {resp.text.strip()!r}')

## Done

Move on to `02-alloydb-graph.ipynb`.